In [1]:
__import__('pysqlite3')
import sys
sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import chromadb

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
files = [
    {
        "title": "Anthropology of Food",
        "source_url": "https://openstax.org/details/books/introduction-anthropology",
        "filename": "anthro_food.txt"
  }
]

In [4]:
# Instantiate a Chroma persistent client
client = chromadb.PersistentClient("./")

collection = client.get_or_create_collection(name='RAG_Assistant', metadata={'hnsw:space':'cosine'})

In [6]:
#Read first file content
with open(f"./{files[0]['filename']}", "r") as file:
  content = file.read()

# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(separators=["\n\n", "\n", ". ", "? ", "! "], 
                                               chunk_size=1500, chunk_overlap=200)

# Split the 'content' into chunks
chunks = text_splitter.create_documents([content])

# Print the first document
chunks[:1]

[Document(metadata={}, page_content='INTRODUCTION\nCHAPTER 14\nAnthropology of Food\n14.1 Food as a Material Arti\ue005act\n14.2 A Biocultural Approach to Food\n14.3 Food and Cultural Identity\n14.4 The Globalization o\ue005 Food\nThe study o\ue005 \ue005ood has a long history in anthropology and weaves together various sub\ue061elds\no\ue005 the discipline. Among other things, \ue005ood connects to nutrition and health, rituals and behaviors regarding\nproduction and consumption, and worldwide trade networks and the related di\ue005\ue005usion o\ue005 plants, animals, and\narti\ue005acts. Distinguishing between what is and what is not \ue005ood is a major concern within and across most\nhuman cultures. Food varies not only \ue005rom one society to another but also across genders, classes, \ue005amily\ngroups, and seasons. As both a source o\ue005 sustenance \ue005or the body and a means o\ue005 establishing or advertising\none’s social status, \ue005ood plays a major role in personal 

In [7]:
# Create empty lists to store each document, metadata, and id
documents = []
metadatas = []
ids = []

# Loop through each file in files
for file_info in files:
    with open(f"./{file_info['filename']}", "r") as file:
        content = file.read()
        # Use text_splitter to create documents
        chunks = text_splitter.create_documents([content])
        # iterate over every chunk
        for index, chunk in enumerate(chunks):
            #Append to metadata list with "title", "source_url", and "index"
            metadatas.append({
                "title": file_info["title"],
                "source_url": file_info["source_url"],
                "chunk_idx": index
            })
            # Append to ids each index
            ids.append(f"{file_info['filename']}_{index}")
            
            # Append to documents each chunk.page_content
            documents.append(chunk.page_content)

In [8]:
# Add all documents to the collection
collection.add(documents=documents, metadatas=metadatas, ids=ids)

# Verify documents were added to collection with a sample query
collection.query(query_texts=['What is the relationship between food and cultural identity.'], n_results=1)

{'ids': [['anthro_food.txt_0']],
 'embeddings': None,
 'documents': [['INTRODUCTION\nCHAPTER 14\nAnthropology of Food\n14.1 Food as a Material Arti\ue005act\n14.2 A Biocultural Approach to Food\n14.3 Food and Cultural Identity\n14.4 The Globalization o\ue005 Food\nThe study o\ue005 \ue005ood has a long history in anthropology and weaves together various sub\ue061elds\no\ue005 the discipline. Among other things, \ue005ood connects to nutrition and health, rituals and behaviors regarding\nproduction and consumption, and worldwide trade networks and the related di\ue005\ue005usion o\ue005 plants, animals, and\narti\ue005acts. Distinguishing between what is and what is not \ue005ood is a major concern within and across most\nhuman cultures. Food varies not only \ue005rom one society to another but also across genders, classes, \ue005amily\ngroups, and seasons. As both a source o\ue005 sustenance \ue005or the body and a means o\ue005 establishing or advertising\none’s social status, \ue005o